**Necessary imports**

In [1]:
import subprocess
import os
import pandas as pd

%load_ext autoreload
%autoreload 2

**Setting parameters**

In [5]:
simple_render_raw_data_dir = "/dlabdata1/tsoares/wikidumps/"
processed_data_dir = "/dlabdata1/boulenge/wikidumps_data/report_examples_3"
server_out_dir = "/var/www/html/wikinsert/report_examples_3"
processes = 48
max_samples_per_lang = 15
langs = ['fr','es', 'simple', 'de'] 

#For all possible languages
#sorted(set([filename.split("wiki")[0] for filename in os.listdir(simple_render_raw_data_dir) if "wiki" in filename]))

side_context_sizes = [5]  #List of context size to try
second_month = "20231001"
third_month = "20231101"

langs_to_remove = ["war", "ky", "nds-nl"] #These languages do not work

for lang in langs_to_remove:
    if lang in langs:
        langs.remove(lang)



**Defining data paths**

In [6]:
# Define directories
rev_files_dir = os.path.join(processed_data_dir, "revisions_files")
parsed_htmls_dir = os.path.join(processed_data_dir, "parsed_htmls")
contexts_splitted_data_dir = os.path.join(processed_data_dir, "contexts_splitted_data")
ready_to_score_data_dir = os.path.join(processed_data_dir, "ready_to_score_data")
scored_data_dir = os.path.join(processed_data_dir, "scored_data")
crawling_output_dir = os.path.join(server_out_dir, "crawled_pages")

# Create necessary directories if they don't exist
os.makedirs(rev_files_dir, exist_ok=True)
os.makedirs(parsed_htmls_dir, exist_ok=True)
os.makedirs(contexts_splitted_data_dir, exist_ok=True)
os.makedirs(ready_to_score_data_dir, exist_ok=True)
os.makedirs(scored_data_dir, exist_ok=True)
os.makedirs(crawling_output_dir, exist_ok=True)
os.makedirs(server_out_dir, exist_ok=True)

**Prepare files to be crawled and parsed**

It generates the diffenret pages ids we will crawl, using data from the previous study

In [7]:
for lang in langs:
        print(f"Processing language: {lang}")
        second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
        third_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{third_month}"
        save_path = f"{rev_files_dir}/revisions_{lang}.parquet"
            
        subprocess.run([
            "python", "generate_revisions_file.py",
            "--versions_file", f"{second_month_path}/eval/link_versions",
            "--html_dir", f"{second_month_path}/eval/pages",
            "--redirect_1", f"{second_month_path}/processed_data/redirect_map.parquet",
            "--redirect_2", f"{third_month_path}/processed_data/redirect_map.parquet",
            "--mention_map", f"{second_month_path}/processed_data/mention_map.parquet",
            "--max_samples", str(max_samples_per_lang),
            "--save_path", save_path
        ])


Processing language: fr


Checking revisions: 100%|██████████| 26847/26847 [01:21<00:00, 330.66it/s]


Processing language: es


Checking revisions: 100%|██████████| 29970/29970 [01:14<00:00, 401.24it/s]


Processing language: simple


Checking revisions: 100%|██████████| 1841/1841 [00:27<00:00, 68.15it/s] 


Processing language: de


Checking revisions: 100%|██████████| 49899/49899 [01:59<00:00, 418.36it/s]


**Crawling first versions only**

There is no need to re-crawl the second version with all its files as the index is sufficent for these.

In [4]:
for lang in langs:
    print(f"Processing language: {lang}")
    subprocess.run([
        "python", "crawl.py",
        "--revisions_path", f"{rev_files_dir}/revisions_{lang}.parquet",
        "--output_dir", f"{crawling_output_dir}/{lang}",
        "--n_processes", str(processes),
        "--lang", lang
    ])

Processing language: fr
Total number of pages to crawl: 10
Running crawling on 48 processes
Crawling finished, post processing...


Moving files: 100%|██████████| 10/10 [00:00<00:00, 3095.88it/s]


Processing language: es
Total number of pages to crawl: 10
Running crawling on 48 processes
Crawling finished, post processing...


Moving files: 100%|██████████| 10/10 [00:00<00:00, 2303.80it/s]


Processing language: en
Total number of pages to crawl: 10
Running crawling on 48 processes
Crawling finished, post processing...


Moving files: 100%|██████████| 10/10 [00:00<00:00, 4755.45it/s]


Processing language: de
Total number of pages to crawl: 10
Running crawling on 48 processes
Crawling finished, post processing...


Moving files: 100%|██████████| 10/10 [00:00<00:00, 3112.89it/s]


**Parsing of the htmls and sentence tokenizing**

In [6]:
for lang in langs:
    print(f"Processing language: {lang}")
    second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
    first_version_specific_html_dir = f"{crawling_output_dir}/{lang}"
    save_path = f"{rev_files_dir}/revisions_{lang}.parquet"
    output_dir = f"{parsed_htmls_dir}/{lang}_v2"
    first_version_specific_output_dir = f"{parsed_htmls_dir}/{lang}_v1"
        
    subprocess.run([
        "python", "parsing.py",
        "--revisions_path", save_path,        
        "--html_dir", f"{second_month_path}/eval/pages",
        "--first_versions_specific_html_dir", first_version_specific_html_dir,
        "--output_dir", output_dir,
        "--first_versions_specific_output_dir", first_version_specific_output_dir,
        "--n_processes", str(processes)
    ])

    parsed_htmls_paths = os.listdir(first_version_specific_output_dir)
    sentences_path = list(filter(lambda x: x.endswith("sentences.jsonl"), parsed_htmls_paths))[0]
    links_path = list(filter(lambda x: x.endswith("links.jsonl"), parsed_htmls_paths))[0]
    sentences_df = pd.read_json(f"{first_version_specific_output_dir}/{sentences_path}")
    links_df = pd.read_json(f"{first_version_specific_output_dir}/{links_path}")

    display(sentences_df.head(5))
    display(links_df.head(5))   

Processing language: fr
First version html dir: /scratch/boulenge/report_examples_2/crawled_pages/fr
Running on 48 processes
Parallel multiprocess
Writing sentences and links v1
Time taken for v1: 16.42017436027527
Writing sentences and links v2


,md5,id,text,startIdx,endIdx,section,locList
0,82297a413d3e029cf14390b48f9d28ff,0,"Pour les articles homonymes, voir Bains (homon...",65242,65382,Lead,"[{'start': 65241, 'end': 65242}, {'start': 652..."
1,fa4f65823512eea09e25a2d66e0c68b1,1,Bains est une commune française située dans le...,80204,80665,Lead,"[{'start': 80203, 'end': 80204}, {'start': 802..."
2,197d52016f861270a066c88e5729240d,2,La commune de Bains se trouve dans le départem...,81508,82048,Géographie,"[{'start': 81507, 'end': 81508}, {'start': 815..."
3,6772b734ee6d2c92f5cde27e9d628600,3,"La commune, traversée par le 45e&nbsp;parallèl...",82049,82521,Géographie,"[{'start': 82047, 'end': 82048}, {'start': 820..."
4,f4384e2eb5674387ed43d71c96e27243,4,Elle se situe à 11&nbsp;km par la route[Note 1...,82522,83892,Géographie,"[{'start': 82520, 'end': 82521}, {'start': 825..."


,link,startIdx,text,endIdx,section,sentid
0,file:///wiki/Projet:Une_pomme_est_un_fruit,34362,Cette page est une très courte ébauche oubliée,34496,Lead,0
1,file:///wiki/Fichier:%C3%89volution_de_l%27art...,34546,Une pomme est un fruit,34685,Lead,1
2,file:///wiki/Wikip%C3%A9dia:Historique_de_Wiki...,34712,histoire de Wikipédia,34873,Lead,1
3,file:///wiki/Wikip%C3%A9dia:N%27h%C3%A9sitez_p...,34875,n'hésitez pas à la développer,34996,Lead,1
4,file:///wiki/Aide:Comment_modifier_une_page,35001,comment ?,35105,Lead,1


Processing language: es
First version html dir: /scratch/boulenge/report_examples_2/crawled_pages/es
Running on 48 processes
Parallel multiprocess
Writing sentences and links v1
Time taken for v1: 78.89932584762573
Writing sentences and links v2


,md5,id,text,startIdx,endIdx,section,locList
0,8f1828f27f18f1e7bd5b63aa67107a60,0,Ilustres ignorantes es un programa de televisi...,55792,56213,Lead,"[{'start': 55791, 'end': 55792}, {'start': 558..."
1,b9422d02d369529d8349e8a095d87d96,1,El programa empezó a emitirse el 1 de noviembr...,56214,56448,Lead,"[{'start': 56212, 'end': 56305}, {'start': 563..."
2,b5b56838921ec75e6d3b52867a88da09,2,Desde el 29 de abril de 2016 se emite en la pl...,56449,56718,Lead,"[{'start': 56447, 'end': 56503}, {'start': 565..."
3,c857327731c26eee41d24e2367652acb,3,Las dos primeras temporadas se grabaron en Vil...,56719,57215,Lead,"[{'start': 56717, 'end': 56873}, {'start': 569..."
4,344755ee8ceef18b9062dd1e599667bc,4,"En cada programa, se presenta a dos invitados ...",57216,57469,Lead,"[{'start': 57214, 'end': 57215}, {'start': 572..."


...

,md5,id,text,startIdx,endIdx,section,locList
0,b618f386f2310c33ab1aff5b8fa03aea,0,Drone attacks on bases in Iraq,47911,47947,Lead,"[{'start': 47911, 'end': 47941}]"
1,7d9ca2c7ba20284ff33e2ab26b876b86,1,"On 18 October 2023, American military bases in...",152970,153285,Lead,"[{'start': 152969, 'end': 152970}, {'start': 1..."
2,6952d1ff6bdf29aa162ddb6cc1e459bb,2,Pentagon officials insisted that there are few...,153286,154190,Lead,"[{'start': 153284, 'end': 153285}, {'start': 1..."
3,d9e733c03098f8e1d9f669c6b3e7d345,3,"On 18 October 2023, amid the 2023 Israel–Hamas...",154334,154978,2023 Al-Asad air base attack,"[{'start': 154333, 'end': 154334}, {'start': 1..."
4,bf85f1d05fb54f50983693b5108c747a,4,"The next day, a false alarm in the airbase cau...",154979,155501,2023 Al-Asad air base attack,"[{'start': 154977, 'end': 154978}, {'start': 1..."


,link,startIdx,text,endIdx,section,sentid
0,file:///wiki/Diary_of_a_Wimpy_Kid:_Rodrick_Rul...,51296,Diary of a Wimpy Kid: Rodrick Rules,51455,Lead,2
1,file:///wiki/Diary_of_a_Wimpy_Kid:_Dog_Days_(f...,51474,Diary of a Wimpy Kid: Dog Days,51608,Lead,2
2,file:///wiki/Supernatural_(American_TV_series),51624,Supernatural,51732,Lead,2
3,file:///wiki/The_Marine_4:_Moving_Target,51748,The Marine 4: Moving Target,51859,Lead,2
4,file:///wiki/Star_Trek_Beyond,51875,Star Trek Beyond,51953,Lead,2


Processing language: de
First version html dir: /scratch/boulenge/report_examples_2/crawled_pages/de
Running on 48 processes
Parallel multiprocess
Writing sentences and links v1
Time taken for v1: 109.60469245910645
Writing sentences and links v2


,md5,id,text,startIdx,endIdx,section,locList
0,718a519b7ed105dae98468cc8729badd,0,"Eine gesichtete Version dieser Seite, die am 28.",9602,9781,Lead,"[{'start': 9602, 'end': 9607}, {'start': 9731,..."
1,d848d35888d13effef3c255400ace663,1,"Dezember 2018 freigegeben wurde, basiert auf d...",9782,10477,Lead,"[{'start': 9781, 'end': 9795}, {'start': 9799,..."
2,67f0022914d3e26b7925f3e879337f65,2,Sa'd allah Abd al-Qadir Lutfi al-Dschabiri (ar...,10478,11132,Lead,"[{'start': 10476, 'end': 10477}, {'start': 104..."
3,de02c1f7320f289db5b6fd137b105ca6,3,Der Sunnit Dschabiri wurde im damals osmanisch...,11133,11250,Lead,"[{'start': 11131, 'end': 11132}, {'start': 111..."
4,829f96938fa93c8ce94b65d1fa664ba1,4,Er hatte fünf Geschwister und besuchte in Alep...,11251,11346,Lead,"[{'start': 11249, 'end': 11345}]"


,link,startIdx,text,endIdx,section,sentid
0,file:///wiki/5._Juni,9658,5. Juni,9709,Lead,2
1,file:///wiki/1905,9710,1905,9752,Lead,2
2,file:///wiki/London,9756,London,9804,Lead,2
3,file:///wiki/24._Mai,9808,24. Mai,9859,Lead,3
4,file:///wiki/1996,9860,1996,9902,Lead,3


**Preprocessing**

It included all prperocessing parts but the merging

In [10]:
for lang in langs:
    for side_context_size in side_context_sizes:
        print(f"Processing language: {lang} with side context size {side_context_size}")
        second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
        third_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{third_month}"
        save_path = f"{rev_files_dir}/revisions_{lang}.parquet"
        output_dir = f"{contexts_splitted_data_dir}/{lang}/context_size_{side_context_size}"
        
        subprocess.run([
            "python", "preprocessing.py",
            "--revisions_path", save_path,
            "--parsed_html_dir", f"{parsed_htmls_dir}/{lang}_v2",
            "--first_versions_specific_parsed_html_dir", f"{parsed_htmls_dir}/{lang}_v1",
            "--n_processes", "1",#str(processes),
            "--output_dir", output_dir,
            "--side_contexts_size", str(side_context_size),
            "--redirect_1", f"{second_month_path}/processed_data/redirect_map.parquet",
            "--redirect_2", f"{third_month_path}/processed_data/redirect_map.parquet"
        ])

        #preprocessed_data_path = os.path.join(output_dir, f'test_data.parquet')

        #preprocessed_data = pd.read_parquet(preprocessed_data_path)
        #display(preprocessed_data.head(5))

Processing language: fr with side context size 5
Running on 1 processes


100%|██████████| 24/24 [00:00<00:00, 4477.11it/s]


The length of the output is 24
Processing language: es with side context size 5
Running on 1 processes


100%|██████████| 21/21 [00:00<00:00, 2757.42it/s]


The length of the output is 21
Processing language: en with side context size 5
Running on 1 processes


100%|██████████| 18/18 [00:00<00:00, 6028.22it/s]


The length of the output is 18
Processing language: de with side context size 5
Running on 1 processes


100%|██████████| 670/670 [00:00<00:00, 6889.19it/s]


The length of the output is 670


**Merging**

In [11]:
test_data_dir = contexts_splitted_data_dir
output_dir = ready_to_score_data_dir

subprocess.run([
    "python", "merge_data.py",
    "--test_data_dir", test_data_dir,
    "--raw_data_dir", simple_render_raw_data_dir,
    "--langs", ",".join(langs),
    "--side_contexts_sizes", ",".join(map(str, side_context_sizes)),
    "--output_dir", output_dir,
    "--ignore_warnings"
])

first_lang = langs[0]
first_context_size = str(side_context_sizes[0])
scored_files_example = os.listdir(output_dir)
scored_files_example = list(filter(lambda x: (first_lang in x and first_context_size in x), scored_files_example))
final_input_data_path = os.path.join(output_dir, scored_files_example[0])

final_input_data_df = pd.read_parquet(final_input_data_path)
display(final_input_data_df.head(5))

Saving mention maps
Processing language fr (1/4)
Finished processing language fr
Processing language es (2/4)
Finished processing language es
Processing language en (3/4)
Finished processing language en
Processing language de (4/4)
Finished processing language de
Processing language fr (1/4)
Loaded pages
Loaded redirects


100%|██████████| 2515482/2515482 [00:08<00:00, 287297.63it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 24 links for language fr
Removed 0 links because of empty contexts
Removed 4 links because target title is not in source titles
Removed 5 links because of missing section
After cleaning, there are 15 links for language fr
Finished processing language fr
Processing language es (2/4)
Loaded pages
Loaded redirects


100%|██████████| 1829995/1829995 [00:07<00:00, 260146.48it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 21 links for language es
Removed 0 links because of empty contexts
Removed 5 links because target title is not in source titles
Removed 1 links because of missing section
After cleaning, there are 15 links for language es
Finished processing language es
Processing language en (3/4)
Loaded pages
Loaded redirects


100%|██████████| 6708565/6708565 [00:32<00:00, 206511.07it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 18 links for language en
Removed 0 links because of empty contexts
Removed 1 links because target title is not in source titles
Removed 0 links because of missing section
After cleaning, there are 17 links for language en
Finished processing language en
Processing language de (4/4)
Loaded pages
Loaded redirects


100%|██████████| 2826417/2826417 [00:10<00:00, 275864.01it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 670 links for language de
Removed 0 links because of empty contexts
Removed 156 links because target title is not in source titles
Removed 443 links because of missing section
After cleaning, there are 71 links for language de
Finished processing language de


,source_title,source_ID,target_title,context,insert_indices,moved_insertion_marker,link_sentence,link_text,section,mention_present,source_lead,first_version,second_version,direct_match,missing_category,all_contexts,target_lead
0,Artillerie,57485,Arbal%C3%A8te,None,[21],None,L’arbalète et la baliste sont apparues pendant...,arbalète,Histoire,None,On appelle artillerie l'ensemble des armes col...,208533468,208533787,None,missing_mention_with_sentence_modifications,[{'context': 'On appelle artillerie l'ensemble...,"L'arbalète est une arme de trait, constituée d..."
1,Artillerie,57485,Baliste,None,[21],None,L’arbalète et la baliste sont apparues pendant...,baliste,Histoire,None,On appelle artillerie l'ensemble des armes col...,208533468,208533787,None,missing_mention_with_sentence_modifications,[{'context': 'On appelle artillerie l'ensemble...,La baliste était un engin de siège développé à...
2,Artillerie,57485,Mausol%C3%A9e_de_l%27empereur_Qin,None,[24],None,Des exemplaires de ces «&nbsp;armes du diable&...,armée de terre cuite,Histoire,None,On appelle artillerie l'ensemble des armes col...,208533468,208533787,None,missing_mention_with_sentence_modifications,[{'context': 'On appelle artillerie l'ensemble...,Le mausolée de l'empereur Qin se trouve à prox...
3,Artillerie,57485,Qin_Shi_Huang,None,[24],None,Des exemplaires de ces «&nbsp;armes du diable&...,Qin shi Huang,Histoire,None,On appelle artillerie l'ensemble des armes col...,208533468,208533787,None,missing_mention_with_sentence_modifications,[{'context': 'On appelle artillerie l'ensemble...,"Qin Shi Huang, ou Qin Shi Huangdi, fut d'abord..."
4,Artillerie,57485,Chu_ko_nu,None,[25],None,"J.-C.), dont une version améliorée pour tirer ...",chu ko nu,Histoire,None,On appelle artillerie l'ensemble des armes col...,208533468,208533787,None,missing_mention_with_sentence_modifications,[{'context': 'On appelle artillerie l'ensemble...,La chu ko nu ou arbalète à répétition est une ...


**Scoring the contexts**

In [28]:
data_dir = ready_to_score_data_dir
output_dir = scored_data_dir

for lang in langs:
    for side_context_size in side_context_sizes:
        print(f"Processing language: {lang} with side context size {side_context_size}")

        subprocess.run([
            "python", "get_llm_scores.py",
            "--data_dir", data_dir,
            "--output_dir", output_dir,
            "--lang", lang,
            "--side_contexts_size", str(side_context_size),
            "--model_dir", "/dlabdata1/tsoares/models",
            "--model_prefix", "roberta_full",
            "--use_section_title",
            "--batch_size", "256",
            "--use_mentions",
            "--use_cuda",
            "--multilingual",
            "--multilingual_name", "multilingual-all"
        ])

scored_files_example = os.listdir(output_dir)[0]
scored_data = pd.read_parquet(f"{output_dir}/{scored_files_example}")

exemple = scored_data['all_contexts'][0][0]

for key in exemple.keys():
    print(f"{key}:\n{exemple[key]}\n")

Processing language: fr with side context size 5
Processing context size 5


100%|██████████| 17/17 [02:45<00:00,  9.73s/it]


Processing language: es with side context size 5
Processing context size 5


100%|██████████| 69/69 [11:11<00:00,  9.73s/it]


Processing language: en with side context size 5
Processing context size 5


100%|██████████| 3/3 [00:25<00:00,  8.50s/it]


Processing language: de with side context size 5
Processing context size 5


100%|██████████| 95/95 [15:25<00:00,  9.75s/it]


context:
On appelle artillerie l'ensemble des armes collectives ou lourdes — ou des troupes qui mettent en œuvre ces armes — servant à envoyer, à courte, moyenne ou grande distance, sur l'ennemi ou sur ses positions et ses équipements, divers projectiles de moyen ou gros calibre&nbsp;: obus, boulet, roquette, missile, pour appuyer ses propres troupes engagées dans une bataille terrestre, navale, antiaérienne ou un siège. Le terme serait apparu vers le XIIIe&nbsp;siècle, dérivant d’un mot du vieux français «&nbsp;artillier&nbsp;» qui servait à désigner les artisans fabricants d'armes et équipements de guerre. Ces artisans ont été pendant longtemps les seuls spécialistes dans le service de ces armes puisqu'ils les fabriquaient et les essayaient avant livraison. C'est pourquoi, jusqu'au XVIIIe&nbsp;siècle, ils étaient commissionnés par les souverains pour servir ces armes à la guerre. Canon exposé devant la basilique Notre-Dame de Montréal, Québec. Ainsi et par extension, le nom d'artille

**Colouring the pages**

In [36]:
output_dir = os.path.join(server_out_dir, "colored_pages")

# Run the coloring script
subprocess.run([
    "python", "color_pages.py",
    "--scored_data_dir", scored_data_dir,
    "--html_dir", crawling_output_dir,
    "--files_link_output_dir", output_dir,
    "--langs", ",".join(langs),
    "--side_contexts_sizes", ",".join(map(str, side_context_sizes)),
    "--extra_crawled_files_dir", crawling_output_dir,
    "--full_model_name", "roberta_full_multilingual-all"
])


Processing language fr with side contexts size 5
Coloring with grey_to_red
Smoothing param: 1
Style: highlight


100%|██████████| 15/15 [00:02<00:00,  5.17it/s]


Processing language es with side contexts size 5
Coloring with grey_to_red
Smoothing param: 1
Style: highlight


100%|██████████| 15/15 [00:06<00:00,  2.17it/s]


Processing language en with side contexts size 5
Coloring with grey_to_red
Smoothing param: 1
Style: highlight


...

Processing language de with side contexts size 5
Coloring with grey_to_red
Smoothing param: 1


  0%|          | 0/71 [00:00<?, ?it/s]

Style: highlight


100%|██████████| 71/71 [00:17<00:00,  4.03it/s]


CompletedProcess(args=['python', 'color_pages.py', '--scored_data_dir', '/dlabdata1/boulenge/wikidumps_data/report_examples_2/scored_data', '--html_dir', '/scratch/boulenge/report_examples_2/crawled_pages', '--files_link_output_dir', '/scratch/boulenge/report_examples_2/colored_pages', '--langs', 'fr,es,en,de', '--side_contexts_sizes', '5', '--extra_crawled_files_dir', '/scratch/boulenge/report_examples_2/crawled_pages', '--full_model_name', 'roberta_full_multilingual-all'], returncode=0)